# 03 -- Stage 3: Merge Image + Video Datasets

Combines `labels.csv` (Stage 1) and `dfdc_frame_manifest.csv`
(Stage 2) into a single manifest, then retrains on the union.

Reads/writes under `dataset_root` (Drive on Colab), not the local
session. If you transferred DFDC frames from Kaggle as a zip bundle
(`dfdc_frames_bundle.zip`, containing `resized_frames/*.jpg` +
`dfdc_frame_manifest.csv`) rather than already-extracted files, upload
that zip to `dataset_root` and this notebook extracts/merges it
automatically below -- no separate script or cell ordering to manage.

In [ ]:
import sys, os

IS_COLAB = os.path.exists("/var/colab/hostname")
IS_KAGGLE = (not IS_COLAB) and (os.path.exists("/kaggle/input") or "KAGGLE_KERNEL_RUN_TYPE" in os.environ)
PROJECT_ROOT = "/kaggle/working/ForensicAI_Lab" if IS_KAGGLE else "/content/ForensicAI_Lab"

if not os.path.exists(PROJECT_ROOT):
    !git clone https://github.com/MRoy03/Forensic_LAB.git {PROJECT_ROOT}

sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)

from notebooks.utils.colab_common import mount_drive, install_requirements, is_colab

if is_colab():
    mount_drive()
    install_requirements(PROJECT_ROOT)
elif IS_KAGGLE:
    !pip install -q -r requirements-train.txt
else:
    print("Not in Colab/Kaggle -- assuming requirements-train.txt is already installed locally.")

In [ ]:
import pandas as pd
from pathlib import Path
from config.loader import get_settings, resolve_dataset_root, resolve_path

settings = get_settings()
dataset_root = resolve_dataset_root(settings)
print("Dataset root:", dataset_root)

## Extract the DFDC frame bundle, if present

Keeps extracted frames on **local Colab disk**, not Drive -- moving
(or later, repeatedly reading during training) 50,000 individual small
files over Drive's FUSE mount is extremely slow in practice (measured:
a run got stuck for hours on this exact step before this fix). Only
the final checkpoint needs real Drive persistence; the extracted raw
frames are cheap to regenerate locally each fresh session.

Skips cleanly if already extracted earlier in this same session, or if
no zip bundle is found (falls through to an image-only merge below).

In [ ]:
import zipfile, shutil

zip_path = dataset_root / "dfdc_frames_bundle.zip"
local_synthetic_dir = Path("/content/local_synthetic_data") if IS_COLAB else Path("/tmp/local_synthetic_data")
local_frame_manifest_path = Path("/content/dfdc_frame_manifest_local.csv") if IS_COLAB else Path("/tmp/dfdc_frame_manifest_local.csv")

if local_frame_manifest_path.exists():
    print(f"{local_frame_manifest_path} already exists -- skipping zip extraction.")
elif zip_path.exists():
    extract_to = Path("/content/dfdc_extract_tmp") if IS_COLAB else Path("/tmp/dfdc_extract_tmp")
    if extract_to.exists():
        shutil.rmtree(extract_to)
    extract_to.mkdir(parents=True)

    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(extract_to)

    local_synthetic_dir.mkdir(parents=True, exist_ok=True)
    for jpg in (extract_to / "resized_frames").glob("*.jpg"):
        shutil.move(str(jpg), str(local_synthetic_dir / jpg.name))

    frame_df = pd.read_csv(extract_to / "dfdc_frame_manifest.csv")
    frame_df["filename"] = frame_df["filename"].apply(lambda p: str(local_synthetic_dir / Path(p).name))
    frame_df.to_csv(local_frame_manifest_path, index=False)

    shutil.rmtree(extract_to)
    print(f"Extracted {len(frame_df)} frames to local disk: {local_synthetic_dir}")
else:
    print(f"No {zip_path} found -- the merge below will proceed with image data only. "
          "Upload dfdc_frames_bundle.zip to this path on Drive first if you want video data included.")

frame_manifest_path = local_frame_manifest_path

## Merge into one manifest

`merged_labels.csv` is saved to Drive for persistence, but video frame
rows reference local-disk paths valid only for *this* session -- if you
resume in a new session later, just re-run the extraction cell above
(fast, local-only) before this one to regenerate valid paths.

In [ ]:
image_labels = pd.read_csv(dataset_root / settings.data.labels_csv)

if frame_manifest_path.exists():
    frame_labels = pd.read_csv(frame_manifest_path)
    missing = (~frame_labels["filename"].apply(lambda p: Path(p).exists())).sum()
    if missing:
        print(f"[WARNING] {missing} of {len(frame_labels)} frame files don't exist -- "
              "check the extraction cell above completed correctly.")

    merged = pd.concat([image_labels, frame_labels], ignore_index=True)
    print(f"Merged {len(image_labels)} image rows + {len(frame_labels)} video frame rows")
else:
    merged = image_labels
    print(f"[WARNING] {frame_manifest_path} not found -- merging image data only.")

merged_path = dataset_root / "merged_labels.csv"
merged.to_csv(merged_path, index=False)
print("Merged manifest:", merged_path, "--", len(merged), "rows")

## Bring checkpoints from Drive into this local session

Checkpoints live under the local session's `models/` folder by
convention (same as Stage 1/02b), not under `dataset_root` -- copy
whatever exists on Drive in so warm-starting below can find them.

In [ ]:
import shutil

drive_models_dir = Path(settings.data.root_drive).parent / "models"
local_models_dir = resolve_path("models")
local_models_dir.mkdir(parents=True, exist_ok=True)

for ckpt_name in ("image_model.pth", "video_model.pth"):
    drive_ckpt = drive_models_dir / ckpt_name
    local_ckpt = local_models_dir / ckpt_name
    if local_ckpt.exists():
        print(f"{ckpt_name} already present locally")
    elif drive_ckpt.exists():
        shutil.copy(drive_ckpt, local_ckpt)
        print(f"Copied {ckpt_name} from Drive to local session")
    else:
        print(f"{ckpt_name} not found on Drive at {drive_ckpt}")

## Validate the merged dataset before training

Same lesson as Stage 1/02b: catch unreadable files up front instead of
crashing mid-epoch. Handles both plain relative filenames (image rows)
and already-absolute paths (remapped video frame rows) the same way.

In [ ]:
import cv2

images_dir = dataset_root / settings.data.images_dir
masks_dir = dataset_root / settings.data.masks_dir

_check_df = pd.read_csv(merged_path)
_bad_indices = []

for i, row in _check_df.iterrows():
    p = Path(row["filename"])
    img_path = p if p.is_absolute() else images_dir / p
    if cv2.imread(str(img_path)) is None:
        _bad_indices.append(i)

print(f"{len(_bad_indices)} of {len(_check_df)} rows failed to load.")
if _bad_indices:
    _check_df = _check_df.drop(index=_bad_indices).reset_index(drop=True)
    _check_df.to_csv(merged_path, index=False)
    print(f"Removed {len(_bad_indices)} bad rows -- merged_labels.csv now has {len(_check_df)} rows.")

## Retrain on the merged dataset

Fixes carried over from the Stage 2 tuning lessons, refined through
several rounds of observed results:
1. **Minority-class oversampling** (`WeightedRandomSampler`) -- at
   ~93% fake / 7% real, a batch_size=32 batch averages under 3 real
   samples, too few for stable gradient signal. Oversampling the
   minority class so every batch sees a balanced mix fixes this.
2. **Plain (unweighted) loss, deliberately** -- an earlier attempt
   combined this sampler *with* class-weighted loss, which
   double-corrected and overshot in the opposite direction in
   practice (`real_acc` 84.86% / `fake_acc` 23.11%). Use the sampler
   *or* loss weighting, not both at once.
3. **Training-only augmentation + sqrt-weighted sampling** -- sampler
   alone, with full inverse-frequency weights, was tried next and
   produced a *worse* failure in the 02b notebook: `real_acc` collapsed
   epoch over epoch (39%->6%) while `train_loss` plummeted, because the
   absolute pool of unique real images is small enough that full
   inverse-frequency weighting repeats each one many times per epoch as
   pixel-identical copies -- the model memorizes those specific images
   instead of learning general real/fake signal. Fixed by (a) applying
   random flip + brightness/contrast jitter to training samples only
   (never validation), so repeats aren't identical, and (b) using
   sqrt-of-inverse-frequency sampling weights, a milder rebalance than
   full inverse frequency.
4. **Held-out validation split + best-checkpoint-by-val-loss saving**,
   plus per-class (real/fake) accuracy and balanced accuracy -- overall
   accuracy alone is misleading on data this imbalanced.

`WARM_START_FROM` defaults to `video_model.pth` -- if `real_acc`/`fake_acc`
are still unbalanced after this fix, also try `"image"` (Stage 1's
checkpoint was trained on much more balanced data, so it may not carry
forward the same real/fake bias as `video_model.pth`).

In [ ]:
WARM_START_FROM = "video"  # "video" or "image"
VAL_FRACTION = 0.15
NUM_EPOCHS = 5

In [ ]:
import random
import torch, numpy as np
from torch.utils.data import Dataset, DataLoader
from services.preprocessing.pipeline import preprocess_image
from services.detection.model import ForensicMultiHeadNet

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Training on:", device)


class ForensicImageDataset(Dataset):
    """augment=True applies a random horizontal flip + mild brightness/
    contrast jitter. Needed specifically because the minority (real)
    class gets oversampled by the sampler below -- without augmentation
    those repeats are pixel-identical, which the model just memorizes
    instead of learning general real/fake signal (this is exactly what
    caused real_acc to collapse over successive epochs in the 02b
    notebook's sampler-only run). Set False for validation -- no
    augmentation there, so reported metrics stay an honest read on
    genuinely-unseen data.
    """

    def __init__(self, labels_csv, images_dir, masks_dir, image_size, augment=False):
        self.df = pd.read_csv(labels_csv)
        self.images_dir = images_dir
        self.masks_dir = masks_dir
        self.image_size = image_size
        self.augment = augment
        self.label_to_idx = {l: i for i, l in enumerate(settings.labels.binary)}
        manip_classes = settings.labels.manipulation_types
        self.manip_to_idx = {m: i for i, m in enumerate(manip_classes)}
        self.none_type = settings.labels.manipulation_type_none

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        p = Path(row["filename"])
        img_path = p if p.is_absolute() else self.images_dir / p
        image_bgr = cv2.imread(str(img_path))
        if image_bgr is None:
            print(f"[WARN] Could not read {img_path}, substituting another sample")
            return self[(idx + 1) % len(self)]

        if self.augment:
            if random.random() < 0.5:
                image_bgr = cv2.flip(image_bgr, 1)
            alpha = random.uniform(0.85, 1.15)  # contrast
            beta = random.uniform(-20, 20)  # brightness
            image_bgr = cv2.convertScaleAbs(image_bgr, alpha=alpha, beta=beta)

        x = preprocess_image(image_bgr, tuple(self.image_size))

        label_idx = self.label_to_idx[row["label"]]
        row_type = row["type"]
        # "none" isn't one of the 6 manipulation_types classes, so .get()
        # would silently default it to class 0 (copy_move) and train the
        # manip head against a wrong label on every real sample -- has_manip=0
        # excludes those from manip_loss instead (same pattern as has_mask).
        has_manip = 0.0 if row_type == self.none_type else 1.0
        manip_idx = self.manip_to_idx.get(row_type, 0)

        mask_path = self.masks_dir / img_path.name
        if mask_path.exists():
            mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
            mask = cv2.resize(mask, tuple(self.image_size))
            mask = (mask > 127).astype(np.float32)[np.newaxis, :, :]
            has_mask = 1.0
        else:
            mask = np.zeros((1, *self.image_size), dtype=np.float32)
            has_mask = 0.0

        return (
            torch.from_numpy(x),
            torch.tensor(label_idx, dtype=torch.long),
            torch.tensor(manip_idx, dtype=torch.long),
            torch.from_numpy(mask),
            torch.tensor(has_mask, dtype=torch.float32),
            torch.tensor(has_manip, dtype=torch.float32),
        )

In [ ]:
label_counts = _check_df["label"].value_counts()
print("Label distribution:\n", label_counts)

label_order = settings.labels.binary
counts = [label_counts.get(l, 0) for l in label_order]
total = sum(counts)
class_weights = torch.tensor(
    [total / (len(counts) * c) if c > 0 else 0.0 for c in counts], dtype=torch.float32
).to(device)
print(f"Class weights ({', '.join(label_order)}):", class_weights.tolist())

In [ ]:
# Split by index first (not torch.utils.data.random_split on one shared
# Dataset instance), so train and val can use separately-configured
# ForensicImageDataset instances -- augmentation must never leak into validation.
manifest_length = len(pd.read_csv(merged_path))
val_size = int(manifest_length * VAL_FRACTION)
train_size = manifest_length - val_size
generator = torch.Generator().manual_seed(42)
perm = torch.randperm(manifest_length, generator=generator).tolist()
train_indices, val_indices = perm[:train_size], perm[train_size:]

train_dataset_full = ForensicImageDataset(merged_path, images_dir, masks_dir, settings.inference.image_size, augment=True)
val_dataset_full = ForensicImageDataset(merged_path, images_dir, masks_dir, settings.inference.image_size, augment=False)
train_dataset = torch.utils.data.Subset(train_dataset_full, train_indices)
val_dataset = torch.utils.data.Subset(val_dataset_full, val_indices)

# Oversample the minority class so each batch sees a more balanced mix,
# but with sqrt-of-inverse-frequency weights rather than full inverse
# frequency -- full inverse frequency forces an exact 50/50 split, which
# at this severity (~93%/7%) means repeating each unique real image many
# times per epoch; combined with augmentation above, sqrt weighting is a
# milder, still-helpful rebalance that avoids near-identical repeats
# dominating the minority class (the failure mode observed in 02b).
train_labels = train_dataset_full.df.iloc[train_indices]["label"]
train_label_counts = train_labels.value_counts()
sample_weights = train_labels.map(lambda l: 1.0 / (train_label_counts[l] ** 0.5)).values
sampler = torch.utils.data.WeightedRandomSampler(
    weights=sample_weights, num_samples=len(sample_weights), replacement=True
)

train_loader = DataLoader(train_dataset, batch_size=32, sampler=sampler, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)
print(f"Train: {len(train_dataset)} rows, Val: {len(val_dataset)} rows")

In [ ]:
model = ForensicMultiHeadNet(
    backbone=settings.models.backbone,
    in_chans=settings.inference.input_channels,
    num_manip_classes=len(settings.labels.manipulation_types),
).to(device)

warm_start_path = local_models_dir / ("video_model.pth" if WARM_START_FROM == "video" else "image_model.pth")
if warm_start_path.exists():
    model.load_state_dict(torch.load(warm_start_path, map_location=device))
    print("Warm-started from", warm_start_path)
else:
    print(f"{warm_start_path} not found -- training from randomly-initialized weights.")

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-4)
# The sampler (previous cell) already balances each batch to ~50/50
# real/fake. Applying class-weighted loss ON TOP of that double-corrects
# and overshoots in the opposite direction (observed in practice:
# real_acc 84.86% / fake_acc 23.11%). Plain unweighted loss now that the
# sampler is doing the rebalancing -- use one or the other, not both.
cls_loss_fn = torch.nn.CrossEntropyLoss()
manip_loss_fn = torch.nn.CrossEntropyLoss(reduction="none")
mask_loss_fn = torch.nn.BCEWithLogitsLoss(reduction="none")
checkpoint_path = resolve_path(settings.models.image_model_checkpoint)
checkpoint_path.parent.mkdir(parents=True, exist_ok=True)

In [ ]:
def compute_loss(out, label_idx, manip_idx, mask, has_mask, has_manip):
    loss_cls = cls_loss_fn(out["cls_logits"], label_idx)
    manip_loss_per_sample = manip_loss_fn(out["manip_logits"], manip_idx)
    loss_manip = (manip_loss_per_sample * has_manip).sum() / (has_manip.sum() + 1e-8)
    mask_loss_per_px = mask_loss_fn(out["mask_logits"], mask)
    loss_mask = (mask_loss_per_px.mean(dim=[1, 2, 3]) * has_mask).sum() / (has_mask.sum() + 1e-8)
    return loss_cls + loss_manip + loss_mask


@torch.no_grad()
def evaluate(loader):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    real_idx = settings.labels.binary.index("real")
    real_correct = real_total = fake_correct = fake_total = 0

    for x, label_idx, manip_idx, mask, has_mask, has_manip in loader:
        x, label_idx, manip_idx = x.to(device), label_idx.to(device), manip_idx.to(device)
        mask, has_mask, has_manip = mask.to(device), has_mask.to(device), has_manip.to(device)
        out = model(x)
        total_loss += compute_loss(out, label_idx, manip_idx, mask, has_mask, has_manip).item()

        preds = out["cls_logits"].argmax(dim=1)
        correct_mask = preds == label_idx
        correct += correct_mask.sum().item()
        total += label_idx.size(0)

        is_real = label_idx == real_idx
        real_total += is_real.sum().item()
        real_correct += (correct_mask & is_real).sum().item()
        fake_total += (~is_real).sum().item()
        fake_correct += (correct_mask & ~is_real).sum().item()

    model.train()
    real_acc = real_correct / real_total if real_total else float("nan")
    fake_acc = fake_correct / fake_total if fake_total else float("nan")
    balanced_acc = (real_acc + fake_acc) / 2
    return total_loss / len(loader), correct / total, real_acc, fake_acc, balanced_acc


best_val_loss = float("inf")
model.train()
for epoch in range(NUM_EPOCHS):
    running_loss = 0.0
    for x, label_idx, manip_idx, mask, has_mask, has_manip in train_loader:
        x, label_idx, manip_idx = x.to(device), label_idx.to(device), manip_idx.to(device)
        mask, has_mask, has_manip = mask.to(device), has_mask.to(device), has_manip.to(device)

        optimizer.zero_grad()
        out = model(x)
        loss = compute_loss(out, label_idx, manip_idx, mask, has_mask, has_manip)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    train_loss = running_loss / len(train_loader)
    val_loss, val_acc, real_acc, fake_acc, balanced_acc = evaluate(val_loader)
    print(f"Epoch {epoch+1}/{NUM_EPOCHS} -- train_loss: {train_loss:.4f}  val_loss: {val_loss:.4f}  "
          f"val_acc: {val_acc:.2%}  real_acc: {real_acc:.2%}  fake_acc: {fake_acc:.2%}  "
          f"balanced_acc: {balanced_acc:.2%}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), checkpoint_path)
        print(f"  -> new best val_loss ({val_loss:.4f}), saved checkpoint to {checkpoint_path}")

print(f"\nBest val_loss: {best_val_loss:.4f}")

## Persist the updated checkpoint to Drive

This overwrites `image_model.pth` going forward -- Stage 4/5/6 continue
from this merged-and-rebalanced checkpoint, not the original Stage 1
image-only one.

In [ ]:
drive_models_dir.mkdir(parents=True, exist_ok=True)
shutil.copy(checkpoint_path, drive_models_dir / checkpoint_path.name)
print("Copied updated checkpoint to", drive_models_dir / checkpoint_path.name)